# Test Dataset Processing
apply the data cleaning and engineering solutions to the test dataset

In [40]:
#Import the required packages
import pandas as pd
import numpy as np
import re

%matplotlib inline


## 1. Read data

In [41]:
# Reading from a csv file, into a data frame with error handling
try:
    # If the csv file has any special symbols, it helps to specify the file encoding.
    df = pd.read_csv('ppr-group-25204989-test.csv', 
                     keep_default_na=True, 
                     delimiter=',', 
                     skipinitialspace=True)
    
    print(f"✓ Successfully loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns")
    
    # Show the first few rows in the data frame
    display(df.head(5))
    
except FileNotFoundError:
    print("ERROR: The file 'ppr-group-25204989-train.csv' was not found.")
    print("Please ensure the file is in the current working directory.")
except Exception as e:
    print(f"ERROR: An unexpected error occurred: {e}")

✓ Successfully loaded dataset with 10000 rows and 9 columns


,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,17/12/2025,"48 NORTHUMBERLAND RD, BALLSBRIDGE, DUBLIN 4",Dublin,D04C6C5,"€2,525,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
1,21/10/2025,"49 ROS NA GREINE, ARDFINNAN, CO TIPPERARY",Tipperary,E91EV70,"€260,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
2,17/12/2025,"39 Beaulieu Banks, Termonfeckin Road, Drogheda",Louth,NaN,"€361,233.00",No,Yes,New Dwelling house /Apartment,NaN
3,30/06/2025,"122 COOLEY RD, DRIMNAGH, DUBLIN 12",Dublin,D12H337,"€435,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
4,14/11/2025,"18 AISLING GAEL, QUINNS CROSS, LIMERICK",Limerick,V94YYW8,"€334,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN


## 2. Data Understading

In [42]:
# Validate that the dataframe has data and expected columns

print("Dataset Validation Checks:")
print("="*60)

# Check 1: Dataset is not empty
if df.empty:
    print("⚠ WARNING: Dataset is empty!")
else:
    print(f"✓ Dataset contains {df.shape[0]:,} rows")

# Check 2: Check for duplicate rows
duplicate_count = df.duplicated().sum()
if duplicate_count > 0:
    print(f"⚠ WARNING: Found {duplicate_count} duplicate rows")
else:
    print("✓ No duplicate rows found")

# Check 3: Check if all columns are unnamed
unnamed_cols = [col for col in df.columns if 'Unnamed' in str(col)]
if unnamed_cols:
    print(f"⚠ WARNING: Found {len(unnamed_cols)} unnamed columns: {unnamed_cols}")
else:
    print("✓ All columns have names")

# Check 4: Basic column overview
print(f"\n✓ Dataset has {df.shape[1]} columns:")
print(f"  - Column names: {list(df.columns)}")

Dataset Validation Checks:
✓ Dataset contains 10,000 rows
⚠ WARNING: Found 2 duplicate rows
✓ All columns have names

✓ Dataset has 9 columns:
  - Column names: ['Date of Sale (dd/mm/yyyy)', 'Address', 'County', 'Eircode', 'Price (€)', 'Not Full Market Price', 'VAT Exclusive', 'Description of Property', 'Property Size Description']


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Date of Sale (dd/mm/yyyy)  10000 non-null  object 
 1   Address                    10000 non-null  object 
 2   County                     10000 non-null  object 
 3   Eircode                    7487 non-null   object 
 4   Price (€)                  10000 non-null  object 
 5   Not Full Market Price      10000 non-null  object 
 6   VAT Exclusive              10000 non-null  object 
 7   Description of Property    10000 non-null  object 
 8   Property Size Description  0 non-null      float64
dtypes: float64(1), object(8)
memory usage: 703.3+ KB


In [44]:
# Correct feature data types

# Change the price feature as numeric. 

df['Price (€)'] = df['Price (€)'].astype(str).str.replace('€', '', regex=False).str.replace(',', '').astype(float)

# Convert all object columns to categorical
for col in df.select_dtypes('object').columns:
    df[col] = df[col].astype('category')

# Update column groups after type corrections
numeric_columns = df.select_dtypes(['int64', 'float64'])
category_columns = df.select_dtypes('category')

display(numeric_columns.describe().T)
display(category_columns.describe().T)

,count,mean,std,min,25%,50%,75%,max
Price (€),10000.0,446317.155462,1.295146e+06,5500.0,255505.75,350000.0,466960.35,88452722.47
Property Size Description,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,count,unique,top,freq
Date of Sale (dd/mm/yyyy),10000,266,18/12/2025,99
Address,10000,9977,"12 APARTMENT BARLEY COURT, THE MALTINGS, BALLI...",2
County,10000,26,Dublin,3074
Eircode,7487,7466,A123456,6
Not Full Market Price,10000,2,No,9509
VAT Exclusive,10000,2,No,7730
Description of Property,10000,3,Second-Hand Dwelling house /Apartment,7717


## 3. Apply Data Quality Plan
### Step 1: Create a backup of the original data

In [45]:

df_raw = df.copy()

print("="*70)
print("BACKUP CREATED")
print("="*70)
print(f"Original dataset preserved with shape: {df_raw.shape}")
print(f"Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")
print(f"Columns: {list(df_raw.columns)}")
print("="*70)

BACKUP CREATED
Original dataset preserved with shape: (10000, 9)
Rows: 10000, Columns: 9
Columns: ['Date of Sale (dd/mm/yyyy)', 'Address', 'County', 'Eircode', 'Price (€)', 'Not Full Market Price', 'VAT Exclusive', 'Description of Property', 'Property Size Description']


### Step 2: Investigate and Drop Problematic Features

In [46]:
# Step 2a: Verify the problematic features exist and analyse them before dropping
print("Analysis of Features to Drop:")
print("-" * 70)


# Analyze Eircode
print("\n1. Eircode - Missing Data Check:")
zeros_count = (df['Eircode'] == 0).sum()
missing_count = df['Eircode'].isnull().sum()
total_problematic = zeros_count + missing_count
pct_problematic = 100 * total_problematic / len(df)
print(f"   Number of zeros: {zeros_count}")
print(f"   Number of NaN: {missing_count}")
print(f"   Total problematic values: {total_problematic} ({pct_problematic:.1f}%)")

# Analyze Property Size Description
print("\n2. Property Size Description - Missing Data check:")
missing_count = df['Property Size Description'].isnull().sum()
pct_missing = 100 * missing_count / len(df)
print(f"   Number of missing values: {missing_count} ({pct_missing:.1f}%)")
print(f"   Non-missing value counts:")
print(f"   {df['Property Size Description'].value_counts(dropna=False)}")

print("\n" + "="*70)
print("DROPPING FEATURES")
print("="*70)

# Drop the problematic columns
features_to_drop = ['Eircode', 'Property Size Description']
df = df.drop(features_to_drop, axis=1)

print(f"\n✓ Successfully dropped {len(features_to_drop)} features:")
for feat in features_to_drop:
    print(f"  - {feat}")
print(f"\nDataset shape after dropping features: {df.shape}")
print(f"Remaining columns ({df.shape[1]}): {list(df.columns)}")

Analysis of Features to Drop:
----------------------------------------------------------------------

1. Eircode - Missing Data Check:
   Number of zeros: 0
   Number of NaN: 2513
   Total problematic values: 2513 (25.1%)

2. Property Size Description - Missing Data check:
   Number of missing values: 10000 (100.0%)
   Non-missing value counts:
   Property Size Description
NaN    10000
Name: count, dtype: int64

DROPPING FEATURES

✓ Successfully dropped 2 features:
  - Eircode
  - Property Size Description

Dataset shape after dropping features: (10000, 7)
Remaining columns (7): ['Date of Sale (dd/mm/yyyy)', 'Address', 'County', 'Price (€)', 'Not Full Market Price', 'VAT Exclusive', 'Description of Property']


### Step 3: Investigate and Remove Duplicate Rows

In [47]:
print("="*70)
print("Investigation 1: Investigate Duplicate Rows")
print("="*70)

# Count duplicate rows
duplicate_count = df.duplicated().sum()

print(f"Number of duplicate rows detected: {duplicate_count}")

if duplicate_count > 0:
    print("\nDuplicate rows found. Displaying duplicate entries:")
    display(df[df.duplicated(keep=False)])
else:
    print("\nNo duplicate rows found.")

Investigation 1: Investigate Duplicate Rows
Number of duplicate rows detected: 2

Duplicate rows found. Displaying duplicate entries:


,Date of Sale (dd/mm/yyyy),Address,County,Price (€),Not Full Market Price,VAT Exclusive,Description of Property
795,19/02/2025,"APT 16 RUSSET COURT, CHURCHYARD LANE, BALLINTE...",Cork,1595000.0,No,No,Second-Hand Dwelling house /Apartment
1474,19/02/2025,"APT 16 RUSSET COURT, CHURCHYARD LANE, BALLINTE...",Cork,1595000.0,No,No,Second-Hand Dwelling house /Apartment
2476,04/02/2025,"9 HIGGINS PARK, FAIRGREEN, PORTLAOISE",Laois,200000.0,No,No,Second-Hand Dwelling house /Apartment
2634,04/02/2025,"9 HIGGINS PARK, FAIRGREEN, PORTLAOISE",Laois,200000.0,No,No,Second-Hand Dwelling house /Apartment


In [48]:
# Remove duplicates
print("Removing rows with duplicates...")
print(f"Shape before dropping: {df.shape}")

# Get the indices of rows to drop
dup_all_idx = df[df.duplicated(keep=False)].index          
dup_drop_idx = df[df.duplicated()].index                  

print(f"Duplicate groups (all rows) indices: {list(dup_all_idx)}")
print(f"Indices to drop (keeping first occurrence): {list(dup_drop_idx)}")
print(f"Rows to drop: {len(dup_drop_idx)}")
df = df.drop(index=dup_drop_idx)

print(f"Shape after dropping: {df.shape}")
print(f"✓ Successfully removed {len(dup_drop_idx)} duplicate row(s)")

Removing rows with duplicates...
Shape before dropping: (10000, 7)
Duplicate groups (all rows) indices: [795, 1474, 2476, 2634]
Indices to drop (keeping first occurrence): [1474, 2634]
Rows to drop: 2
Shape after dropping: (9998, 7)
✓ Successfully removed 2 duplicate row(s)


### Step 4: Pick out and Drop Bulk Property Transactions

In [49]:
def is_bulk_sale(address):
    if not isinstance(address, str): return False
    addr = address.upper()

    # 1. Detects multiple units or house ranges (e.g., 'HOUSES', 'APARTMENTS', '1-10', '1 TO 5', '28 29 30').
    # These are definitive markers of bulk/portfolio transactions.
    pattern1 = r'BLOCKS|HOUSES|APARTMENTS|UNITS|\d+\s+TO\s+\d+|\d+-\d+|\d+\s+\d+'
    
    # 2. INSTITUTIONAL & LARGE-SCALE DEVELOPMENTS
    # Targets non-residential entities or macro-scale development projects 
    # (e.g., 'UNIVERSITY', 'CAMPUS') where sales involve entire buildings.
    pattern2 = r'CAMPUS|DEVELOPMENT|SCHEME|UNIVERSITY|COLLEGE|HOSPITAL'
    
    # 3. PROXY UNIT & BUILDING ANCHORS
    # Identifies cases where an entire block is registered under the first unit (e.g., 'APT 1')
    # or starts with 'BLOCK', a common indicator for multi-unit assets.
    pattern3 = r'^APT\s+1\b|^UNIT\s+1\b|^SUITE\s+1\b|^HOUSE\s+1\b|^BLOCK'

    # 4. LANDMARK HEADERS (BEFORE FIRST COMMA)
    # Identifies landmarks or estate names appearing before the first comma 
    # without a specific house number, typical for institutional acquisitions.
    pattern4 = r'^[^,]*(SQUARE|CROSS|MILLS)\b'

    if re.search(pattern1, addr): return True
    if re.search(pattern2, addr): return True
    if re.search(pattern3, addr): return True
    if re.search(pattern4, addr): return True
    
    return False

df['is_bulk'] = df['Address'].apply(is_bulk_sale)

df_bulk=df[df['is_bulk'] == True]
df = df.drop(df_bulk.index)
df.drop(columns=['is_bulk'],inplace=True)

print(f"Shape after dropping: {df.shape}")
print(f"✓ Successfully removed {len(df_bulk)} bulk transaction row(s)")

Shape after dropping: (9752, 7)
✓ Successfully removed 246 bulk transaction row(s)


### Step 5: Price Clamp
As price is regarded as unknown, this step is omitted. 


### Step 6: Feature Engineering

#### 1. Datetime extraction

In [50]:
# Convert to datetime (format: dd/mm/yyyy)
df["Date of Sale (dd/mm/yyyy)"] = pd.to_datetime(
    df["Date of Sale (dd/mm/yyyy)"],
    dayfirst=True,
    errors="coerce"
)

print("Date conversion completed.")

# Extract temporal features
df["SaleYear"] = df["Date of Sale (dd/mm/yyyy)"].dt.year
df["SaleMonth"] = df["Date of Sale (dd/mm/yyyy)"].dt.month

print("Year and Month extracted successfully.")

# Convert SaleMonth to categorical for better memory efficiency and analysis
df["SaleMonth"] = df["SaleMonth"].astype("category")

# Drop the original date column as it's no longer needed
df.drop(columns=["Date of Sale (dd/mm/yyyy)"], inplace=True)
print("Original date column removed.")

Date conversion completed.
Year and Month extracted successfully.
Original date column removed.


#### 2. Address extraction

In [51]:
# set the default value as NON_DUBLIN
df['Region'] = 'NON_DUBLIN'

# define the pattern for matching "Dublin ##""D##" and "Dublin"
pattern = r'\b(DUBLIN\s*\d+|D\d+[A-Z]?|DUBLIN)\b'

# select the rows whose county is Dublin and extract districts from addresses
is_dublin_county = df['County'].str.contains('Dublin', case=False, na=False)
extracted = df.loc[is_dublin_county, 'Address'].astype(str).str.extract(pattern, flags=re.IGNORECASE, expand=True)[0]
extracted = extracted.str.upper().str.replace(' ', '', regex=False)
extracted = extracted.replace('DUBLIN', 'DUBLIN_OTHER')
df['Region'] = extracted

# if no result but the county is dublin, fill it as DUBLIN_OTHER
df.loc[(df['Region'].isna()) & (df['County'].str.contains('Dublin', case=False, na=False)), 'Region'] = 'DUBLIN_OTHER'

df.loc[is_dublin_county, 'Region'] = extracted

df.loc[is_dublin_county & df['Region'].isna(), 'Region'] = 'DUBLIN_OTHER'

df['Region'] = df['Region'].fillna('NON_DUBLIN')

print(df['Region'].value_counts())


Region
NON_DUBLIN      6777
DUBLIN_OTHER    1432
DUBLIN15         188
DUBLIN7          108
DUBLIN24         100
DUBLIN9           97
DUBLIN12          89
DUBLIN18          88
DUBLIN8           82
DUBLIN4           81
DUBLIN11          77
DUBLIN3           77
DUBLIN16          76
DUBLIN5           75
DUBLIN14          71
DUBLIN22          68
DUBLIN6           57
DUBLIN13          56
DUBLIN1           51
DUBLIN2           31
DUBLIN20          29
DUBLIN10          27
DUBLIN17          14
DUBLIN08           1
Name: count, dtype: int64


In [52]:
# replace DUBLIN08 with DUBLIN8
df['Region']=df['Region'].replace('DUBLIN08','DUBLIN8')
print(df['Region'].value_counts())

Region
NON_DUBLIN      6777
DUBLIN_OTHER    1432
DUBLIN15         188
DUBLIN7          108
DUBLIN24         100
DUBLIN9           97
DUBLIN12          89
DUBLIN18          88
DUBLIN8           83
DUBLIN4           81
DUBLIN3           77
DUBLIN11          77
DUBLIN16          76
DUBLIN5           75
DUBLIN14          71
DUBLIN22          68
DUBLIN6           57
DUBLIN13          56
DUBLIN1           51
DUBLIN2           31
DUBLIN20          29
DUBLIN10          27
DUBLIN17          14
Name: count, dtype: int64


In [53]:
# Drop the original date column as it's no longer needed
df.drop(columns=["Address"], inplace=True)
print("Original date column removed.")

Original date column removed.


In [54]:
# Standarlize feature names
df.columns = df.columns.str.replace(' ', '')

### Step 6: Validate Cleaned Dataset

In [55]:
print("="*70)
print("TEST DATA QUALITY VALIDATION")
print("="*70)

# Comparison: Before vs After
print("\n1. DATASET DIMENSIONS COMPARISON")
print("-" * 70)
print(f"Original dataset:")
print(f"  Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")
print(f"\nCleaned dataset:")
print(f"  Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"\nChanges:")
print(f"  Rows removed: {df_raw.shape[0] - df.shape[0]} (4 duplictate rows)")
print(f"  Columns dropped: {df_raw.shape[1] - df.shape[1]} (2 problematic features)")

# Check for missing values
print("\n2. MISSING VALUES CHECK")
print("-" * 70)
missing_summary = df.isnull().sum()
if missing_summary.sum() == 0:
    print("✓ All missing values processed successfully - NONE remaining")
else:
    print("⚠ WARNING: Remaining missing values found:")
    print(missing_summary[missing_summary > 0])

# Check data types
print("\n3. DATA TYPES CHECK")
print("-" * 70)
print("Remaining features and their types:")
df.info()


# Check for duplicates
print("\n4. DUPLICATE CHECK")
print("-" * 70)
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")
if dup_count == 0:
    print("✓ No duplicate rows found")
else:
    print(f"⚠ WARNING: {dup_count} duplicate rows detected")

print("\n" + "="*70)
print("VALIDATION COMPLETE")
print("="*70)

TEST DATA QUALITY VALIDATION

1. DATASET DIMENSIONS COMPARISON
----------------------------------------------------------------------
Original dataset:
  Rows: 10000, Columns: 9

Cleaned dataset:
  Rows: 9752, Columns: 8

Changes:
  Rows removed: 248 (4 duplictate rows)
  Columns dropped: 1 (2 problematic features)

2. MISSING VALUES CHECK
----------------------------------------------------------------------
✓ All missing values processed successfully - NONE remaining

3. DATA TYPES CHECK
----------------------------------------------------------------------
Remaining features and their types:
<class 'pandas.core.frame.DataFrame'>
Index: 9752 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   County                 9752 non-null   category
 1   Price(€)               9752 non-null   float64 
 2   NotFullMarketPrice     9752 non-null   category
 3   VATExclusive           9752

In [56]:
# Stats for numeric features.
print(df.dtypes)


County                   category
Price(€)                  float64
NotFullMarketPrice       category
VATExclusive             category
DescriptionofProperty    category
SaleYear                    int32
SaleMonth                category
Region                     object
dtype: object


In [57]:
df['SaleYear']=df['SaleYear'].astype('category')
df.describe(include=['int', 'float']).T

,count,mean,std,min,25%,50%,75%,max
Price(€),9752.0,414009.397078,683061.259189,5500.0,255506.61,350000.0,465000.0,43192661.0


In [58]:
df.describe(include="category").T

,count,unique,top,freq
County,9752,26,Dublin,2975
NotFullMarketPrice,9752,2,No,9280
VATExclusive,9752,2,No,7565
DescriptionofProperty,9752,3,Second-Hand Dwelling house /Apartment,7553
SaleYear,9752,1,2025,9752
SaleMonth,9752,12,12,1002


### Step 6: Save Cleaned Dataset

In [59]:
print("="*70)
print("SAVING CLEANED DATASET")
print("="*70)

# Define output filename with version indicator
output_filename = 'ppr-group-25204989-test-Cleaned.csv'

# Write the cleaned dataframe to a csv file with UTF-8 encoding
df.to_csv(output_filename, index=False, encoding='utf-8')

print(f"\n✓ Cleaned dataset saved successfully")
print(f"\nFile Information:")
print(f"  Filename: {output_filename}")
print(f"  Location: Current working directory")
print(f"  Format: CSV (UTF-8 encoding)")
print(f"  Rows: {df.shape[0]}")
print(f"  Columns: {df.shape[1]}")
print("="*70)

SAVING CLEANED DATASET

✓ Cleaned dataset saved successfully

File Information:
  Filename: ppr-group-25204989-test-Cleaned.csv
  Location: Current working directory
  Format: CSV (UTF-8 encoding)
  Rows: 9752
  Columns: 8


## 3. Apply Feature Engineering Solution
#### Step 1:  SaleYear and SaleMonth
Convert them into a month index

In [60]:
# change data type of saleyear and salemonth
df["SaleYear"] = df["SaleYear"].astype(int)
df["SaleMonth"] = df["SaleMonth"].astype(int)
# create month index starting from 2016-01
df["SaleMonthIndex"] = (df["SaleYear"] - 2016) * 12 + df["SaleMonth"]
# check result
df[["SaleYear", "SaleMonth", "SaleMonthIndex"]].head(10)

,SaleYear,SaleMonth,SaleMonthIndex
0,2025,12,120
1,2025,10,118
2,2025,12,120
3,2025,6,114
4,2025,11,119
5,2025,11,119
6,2025,9,117
7,2025,1,109
8,2025,4,112
9,2025,10,118


#### Step 2: NotFullMarketPrice
Change it into binary number, yes-1, no-0

In [61]:
print(df["NotFullMarketPrice"].unique())
df["NotFullMarketPrice"] = df["NotFullMarketPrice"].map({
    "Yes": 1,
    "No": 0
})
print(df["NotFullMarketPrice"].head())

['No', 'Yes']
Categories (2, object): ['No', 'Yes']
0    0
1    0
2    0
3    0
4    0
Name: NotFullMarketPrice, dtype: category
Categories (2, int64): [0, 1]


#### Step 3: VATExclusive
Change it into binary number, yes-1, no-0

In [62]:
print(df["VATExclusive"].unique())
df["VATExclusive"] = df["VATExclusive"].map({
    "Yes": 1,
    "No": 0
})
print(df["VATExclusive"].head())

['No', 'Yes']
Categories (2, object): ['No', 'Yes']
0    0
1    0
2    1
3    0
4    0
Name: VATExclusive, dtype: category
Categories (2, int64): [0, 1]


#### Step 4: DescriptionofProperty
Change it into two binary feature, new -1，second-hand -0.    

'Teach/Árasán Cónaithe Atháimhe' means second house/apartment in Irish, so we all merge 'Teach/Árasán Cónaithe Atháimhe' into second-hand house.

In [63]:
print(df["DescriptionofProperty"].unique())

['Second-Hand Dwelling house /Apartment', 'New Dwelling house /Apartment', 'Teach/Árasán Cónaithe Atháimhe']
Categories (3, object): ['New Dwelling house /Apartment', 'Second-Hand Dwelling house /Apartment', 'Teach/Árasán Cónaithe Atháimhe']


In [64]:
df["DescriptionofProperty"] = df["DescriptionofProperty"].map({
    "Second-Hand Dwelling house /Apartment": 0,
    "New Dwelling house /Apartment": 1,
    "Teach/Árasán Cónaithe Atháimhe":0
})
print(df["DescriptionofProperty"].head(1000))

0       0
1       0
2       1
3       0
4       0
       ..
1025    1
1026    0
1027    0
1028    1
1029    0
Name: DescriptionofProperty, Length: 1000, dtype: int64


#### Step 5. County and Region encoding
Merge county and region, and encode it with the mean price of 2024 in the region.

In [ ]:
# Firstly, integrate county and region together as region.
df['Region'] = np.where(df['Region'] == 'NON_DUBLIN', df['County'], df['Region'])

df['Region'].value_counts()

Region
DUBLIN_OTHER    1432
Cork            1174
Kildare          546
Galway           464
Meath            441
Wicklow          362
Wexford          343
Louth            323
Limerick         321
Waterford        252
Tipperary        248
Kerry            230
Clare            225
Donegal          219
Mayo             218
Westmeath        204
DUBLIN15         188
Kilkenny         180
Laois            170
Offaly           153
Sligo            131
Cavan            127
Roscommon        124
DUBLIN7          108
DUBLIN24         100
DUBLIN9           97
Carlow            91
DUBLIN12          89
DUBLIN18          88
DUBLIN8           83
DUBLIN4           81
Longford          78
Monaghan          77
DUBLIN3           77
DUBLIN11          77
Leitrim           76
DUBLIN16          76
DUBLIN5           75
DUBLIN14          71
DUBLIN22          68
DUBLIN6           57
DUBLIN13          56
DUBLIN1           51
DUBLIN2           31
DUBLIN20          29
DUBLIN10          27
DUBLIN17          14
Name: 

In [ ]:
# As we need the mean price of 2024, we read in the train dataset.
df_train = pd.read_csv('ppr-group-25204989-train-Engineered.csv', 
                     keep_default_na=True, 
                     delimiter=',', 
                     skipinitialspace=True)
df_train.dtypes

# Pick out the data of 2023 and 2024
df2324=df_train[(df_train['SaleYear'] == 2024) | (df_train['SaleYear'] == 2023)].reset_index(drop=True)
df2324

,County,Price(€),NotFullMarketPrice,VATExclusive,DescriptionofProperty,SaleYear,SaleMonth,Region,YearMonth,SaleMonthIndex,Region_Encoded
0,Dublin,255000.00,0,0,0,2023,12,DUBLIN_OTHER,202312,96,444420.359784
1,Leitrim,262000.00,0,0,0,2023,8,Leitrim,202308,92,153068.540000
2,Dublin,418502.00,0,1,1,2023,3,DUBLIN_OTHER,202303,87,444420.359784
3,Cavan,115000.00,0,0,0,2023,1,Cavan,202301,85,180145.335000
4,Donegal,160000.00,0,0,0,2023,7,Donegal,202307,91,178609.911494
...,...,...,...,...,...,...,...,...,...,...,...
11716,Kildare,189130.00,0,0,0,2024,12,Kildare,202412,108,363792.800982
11717,Clare,110000.00,0,0,0,2024,10,Clare,202410,106,227189.696864
11718,Dublin,440528.63,0,1,1,2024,6,DUBLIN_OTHER,202406,102,455169.126860
11719,Kildare,299559.47,0,1,1,2024,5,Kildare,202405,101,363792.800982


In [ ]:
# Check the samples grouped by region and year both in a single year and in two years that are less than 10.

# pair the region with year
annual_counts = df2324.groupby(['SaleYear', 'Region']).size().reset_index(name='single_year_count')

# count the number of samples of the same region in two yeas.
annual_counts['two_year_total'] = annual_counts.groupby('Region')['single_year_count'].rolling(window=2, min_periods=1).sum().reset_index(level=0, drop=True)

# pick those less than 10 and null values out.
sparse_1y = annual_counts[(annual_counts['single_year_count'] < 10) | (annual_counts['single_year_count'].isna())]
sparse_2y = annual_counts[(annual_counts['two_year_total'] < 10) | (annual_counts['two_year_total'].isna())]

# print region-year pair less than 10 in a single year.
print(f"There are {len(sparse_1y)} year-region pairs whose samples are less than 10.")
print(sparse_1y.sort_values(by='single_year_count').to_string(index=False))

# print region-year pair less than 10 in two years.
if sparse_2y.empty:
    print("All samples in two_year_rolling are more than 10.")
else:
    print(f"There are {len(sparse_2y)} groups whose samples are less than 10 in two_year_rolling")
    print(sparse_2y[['SaleYear', 'Region', 'two_year_total']].sort_values(by='two_year_total').to_string(index=False))


There are 3 year-region pairs whose samples are less than 10.
 SaleYear   Region  single_year_count  two_year_total
     2024 DUBLIN17                  4            14.0
     2023 DUBLIN20                  7             7.0
     2024 DUBLIN20                  9            16.0
There are 1 groups whose samples are less than 10 in two_year_rolling
 SaleYear   Region  two_year_total
     2023 DUBLIN20             7.0


2024-DUBLIN20 2024-DUBLIN17 do not meet the requirement that number of samples >=10 in a single year. But samples of DUBLIN17 and DUBLIN20 in two years (2023 and 2024) are >=10. 
<br><br>

Now start to encoding the region.<br>
First, calculate the average price and count the number of samples grouped by region and year (both in single year and in two years).

In [68]:
# 1. calculate the mean and count sample number in a single year
stats_1y = df2324.groupby(['SaleYear', 'Region'])['Price(€)'].agg(['mean', 'count']).reset_index()

# 2. calculate the mean and count sample number in two years. As 2016 has no prevous year, its two-year values are filled with the value in 2016 only
stats_2y = stats_1y.sort_values(['Region', 'SaleYear']).copy()
stats_2y['rolling_sum'] = stats_2y.groupby('Region')['mean'].transform(lambda x: x * stats_2y['count'])
stats_2y['cum_count_2y'] = stats_2y.groupby('Region')['count'].transform(lambda x: x.rolling(window=2, min_periods=1).sum())
stats_2y['cum_mean_2y'] = stats_2y.groupby('Region')['rolling_sum'].transform(lambda x: x.rolling(window=2, min_periods=1).sum()) / stats_2y['cum_count_2y']

# 3. pick out the mean of DUBLIN_OTHER for later substitution 
stats_dublin_other = df[df['Region'] == 'DUBLIN_OTHER'].groupby('SaleYear')['Price(€)'].mean().reset_index(name='dublin_other_mean')


Second, move the results forward by one year (2024 →2025)

In [69]:
lookup_1y = stats_1y[['SaleYear', 'Region', 'mean', 'count']].copy()
lookup_1y['SaleYear'] += 1

lookup_2y = stats_2y[['SaleYear', 'Region', 'cum_mean_2y', 'cum_count_2y']].copy()
lookup_2y['SaleYear'] += 1

lookup_dublin = stats_dublin_other.copy()
lookup_dublin['SaleYear'] += 1


Third, merge the result into df and fill the encoding.

In [70]:
# 1. merge all the results into the main df
df = df.merge(lookup_1y, on=['SaleYear', 'Region'], how='left', suffixes=('', '_1y'))
df = df.merge(lookup_2y, on=['SaleYear', 'Region'], how='left')
df = df.merge(lookup_dublin, on='SaleYear', how='left')

# 2. set the column of new feature
df['Region_Encoded'] = np.nan
threshold = 10  

# fill the new feature by Hierarchy

# Hierarchy 1: mean of prevous year that has more than 10 samples
mask1 = df['count'] >= threshold
df.loc[mask1, 'Region_Encoded'] = df['mean']

# Hierarchy 2: mean of prevous two years that has more than 10 samples
mask2 = df['Region_Encoded'].isna() & (df['cum_count_2y'] >= threshold)
df.loc[mask2, 'Region_Encoded'] = df['cum_mean_2y']

# Hierarchy 3: for Regions in Dublin which does not meet Hierarchy 2, fill with the mean of DUBLIN_OTHER
dublin_regions = [r for r in df['Region'].unique() if 'Dublin' in r or r.isdigit()]
is_dublin = df['Region'].isin(dublin_regions)
mask3 = df['Region_Encoded'].isna() & is_dublin
df.loc[mask3, 'Region_Encoded'] = df['dublin_other_mean']


# use the total mean to cover the bottom
df['Region_Encoded'] = df['Region_Encoded'].fillna(df['Price(€)'].mean())

# 3. drop the unnecissary columns
cols_to_drop = ['mean', 'count', 'cum_mean_2y', 'cum_count_2y', 'dublin_other_mean']
df.drop(columns=cols_to_drop, inplace=True)

#### Step 6: Validate the engineered data

In [71]:
print(df.dtypes)

County                   category
Price(€)                  float64
NotFullMarketPrice       category
VATExclusive             category
DescriptionofProperty       int64
SaleYear                    int64
SaleMonth                   int64
Region                     object
SaleMonthIndex              int64
Region_Encoded            float64
dtype: object


In [72]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Price(€),9752.0,414009.397078,683061.259189,5500.000000,255506.610000,350000.000000,465000.000000,4.319266e+07
DescriptionofProperty,9752.0,0.225390,0.417860,0.000000,0.000000,0.000000,0.000000,1.000000e+00
SaleYear,9752.0,2025.000000,0.000000,2025.000000,2025.000000,2025.000000,2025.000000,2.025000e+03
SaleMonth,9752.0,6.904635,3.375719,1.000000,4.000000,7.000000,10.000000,1.200000e+01
SaleMonthIndex,9752.0,114.904635,3.375719,109.000000,112.000000,115.000000,118.000000,1.200000e+02
Region_Encoded,9752.0,343187.630221,94825.477578,167792.857143,266697.775259,332809.963642,426834.413543,5.649479e+05


In [73]:

df['Region']=df['Region'].astype('category')
df['NotFullMarketPrice']=df['NotFullMarketPrice'].astype('int64')
df['VATExclusive']=df['VATExclusive'].astype('int64')
df['SaleMonth']=df['SaleMonth'].astype('category')

print(df.dtypes)

County                   category
Price(€)                  float64
NotFullMarketPrice          int64
VATExclusive                int64
DescriptionofProperty       int64
SaleYear                    int64
SaleMonth                category
Region                   category
SaleMonthIndex              int64
Region_Encoded            float64
dtype: object


In [74]:
df.isnull().sum()

County                   0
Price(€)                 0
NotFullMarketPrice       0
VATExclusive             0
DescriptionofProperty    0
SaleYear                 0
SaleMonth                0
Region                   0
SaleMonthIndex           0
Region_Encoded           0
dtype: int64

In [75]:
df.nunique()

County                     26
Price(€)                 2208
NotFullMarketPrice          2
VATExclusive                2
DescriptionofProperty       2
SaleYear                    1
SaleMonth                  12
Region                     47
SaleMonthIndex             12
Region_Encoded             47
dtype: int64

In [76]:
# back up the engineered dataset
df.to_csv('ppr-group-25204989-test-Engineered.csv', index=False, encoding='utf-8')

In [77]:
col_to_drop=df.select_dtypes('category')
df.drop(columns=col_to_drop, inplace=True)
print(df.dtypes)


Price(€)                 float64
NotFullMarketPrice         int64
VATExclusive               int64
DescriptionofProperty      int64
SaleYear                   int64
SaleMonthIndex             int64
Region_Encoded           float64
dtype: object


In [78]:
print("="*70)
print("SAVING DATASET FOR EVALUATION")
print("="*70)

# Define output filename with version indicator
output_filename = 'ppr-group-25204989-test-Forevaluation.csv'

# Write the cleaned dataframe to a csv file with UTF-8 encoding
df.to_csv(output_filename, index=False, encoding='utf-8')

print(f"\n✓ Dataset for evaluation saved successfully")
print(f"\nFile Information:")
print(f"  Filename: {output_filename}")
print(f"  Location: Current working directory")
print(f"  Format: CSV (UTF-8 encoding)")
print(f"  Rows: {df.shape[0]}")
print(f"  Columns: {df.shape[1]}")
print(f"\nDataset is now ready for:")
print(f"  - Evaluation (test the predictive model)")
print("="*70)


SAVING DATASET FOR EVALUATION

✓ Dataset for evaluation saved successfully

File Information:
  Filename: ppr-group-25204989-test-Forevaluation.csv
  Location: Current working directory
  Format: CSV (UTF-8 encoding)
  Rows: 9752
  Columns: 7

Dataset is now ready for:
  - Evaluation (test the predictive model)
